# Plant Leaf Disease Detection
Transfer learning (MobileNetV2) with TensorFlow/Keras.

**Before running:** Runtime -> Change runtime type -> GPU.

Your dataset should be organized like:
```
dataset/
  Tomato_Healthy/
    img1.jpg ...
  Tomato_Blight/
    img1.jpg ...
  ...
```

## 1. Mount Google Drive (if your dataset is stored there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS: path to your dataset folder (the one containing one subfolder per class)
DATASET_DIR = '/content/drive/MyDrive/dataset'

## 2. Imports and config

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS_STAGE1 = 10   # train only the new head
EPOCHS_STAGE2 = 10   # fine-tune top layers of base model

## 3. Load dataset (auto splits train/validation from folder structure)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', class_names)
print('Number of classes:', num_classes)

## 4. Visualize a few sample images

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')

## 5. Performance setup + data augmentation
Augmentation reduces overfitting, which matters a lot on leaf datasets that often have limited images per class.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

## 6. Build the model (MobileNetV2 transfer learning)
MobileNetV2 is small, fast, and works well on Colab's free GPU while still giving strong accuracy.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze for stage 1

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Stage 1: train the new classifier head

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks
)

## 8. Stage 2: fine-tune top layers of the base model
Unfreezes the last portion of MobileNetV2 and trains with a much lower learning rate to squeeze out extra accuracy.

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # unfreeze last 30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

total_epochs = EPOCHS_STAGE1 + EPOCHS_STAGE2
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history1.epoch[-1] + 1,
    callbacks=callbacks
)

## 9. Plot training curves

In [ ]:
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(EPOCHS_STAGE1 - 1, color='gray', linestyle='--', label='Fine-tune start')
plt.legend(); plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(EPOCHS_STAGE1 - 1, color='gray', linestyle='--', label='Fine-tune start')
plt.legend(); plt.title('Loss')
plt.show()

## 10. Evaluate: confusion matrix + classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Save the model

In [ ]:
model.save('/content/plant_disease_model.keras')

# Save class names alongside the model so you can decode predictions later
with open('/content/class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))

# Optional: copy to Drive so it persists after the Colab session ends
# import shutil
# shutil.copy('/content/plant_disease_model.keras', '/content/drive/MyDrive/plant_disease_model.keras')

## 12. Predict on a single new image

In [ ]:
def predict_image(img_path, model=model, class_names=class_names):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    preds = model.predict(img_array, verbose=0)[0]
    top_idx = np.argmax(preds)
    print(f'Predicted: {class_names[top_idx]}  (confidence: {preds[top_idx]*100:.2f}%)')
    for i, name in enumerate(class_names):
        print(f'  {name}: {preds[i]*100:.2f}%')

# Example usage:
# predict_image('/content/drive/MyDrive/dataset/Tomato_Blight/sample.jpg')